# LexAgent v2.0 | Phase 1 | End-to-End Test

**DVA+ Protocol — Hybrid Retrieval Layer**

This notebook validates the complete Phase 1 pipeline:
1. Install dependencies
2. Mount Google Drive
3. Load Mistral-7B (NF4 4-bit)
4. Load Legal-BERT embedder
5. Download legal datasets (LegalBench, CaseHOLD, SCOTUS)
6. Build ChromaDB index
7. Build BM25 index
8. Test hybrid retrieval (RRF fusion)
9. Print system summary

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ══════════════════════════════════════════════════════
# Cell 1: Install Requirements
# ══════════════════════════════════════════════════════
!pip install -q -r /content/drive/MyDrive/LexAgent/lexagent/requirements.txt
print("✅ All dependencies installed.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18

In [6]:
# ══════════════════════════════════════════════════════
# Cell 2: Mount Google Drive & Setup Paths
# ══════════════════════════════════════════════════════
import sys
import os

# Add LexAgent to Python path
sys.path.insert(0, '/content/drive/MyDrive/LexAgent/lexagent')

from src.utils.drive_manager import DriveManager
from config import DRIVE_BASE, CHROMA_CORPUS_PATH, BM25_INDEX_PATH

# Mount Drive and create directories
drive_mgr = DriveManager()
drive_mgr.mount()

print(f"\n✅ Drive mounted. Base path: {DRIVE_BASE}")
print(f"   ChromaDB path: {CHROMA_CORPUS_PATH}")
print(f"   BM25 index:    {BM25_INDEX_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[LexAgent] 2026-06-15 04:58:43 | src.utils.drive_manager | INFO | Google Drive mounted successfully.


INFO:src.utils.drive_manager:Google Drive mounted successfully.


[LexAgent] 2026-06-15 04:58:43 | src.utils.drive_manager | INFO | Directory ensured: /content/drive/MyDrive/LexAgent


INFO:src.utils.drive_manager:Directory ensured: /content/drive/MyDrive/LexAgent


[LexAgent] 2026-06-15 04:58:43 | src.utils.drive_manager | INFO | Directory ensured: /content/drive/MyDrive/LexAgent/chroma_corpus


INFO:src.utils.drive_manager:Directory ensured: /content/drive/MyDrive/LexAgent/chroma_corpus


[LexAgent] 2026-06-15 04:58:43 | src.utils.drive_manager | INFO | Directory ensured: /content/drive/MyDrive/LexAgent/chroma_memory


INFO:src.utils.drive_manager:Directory ensured: /content/drive/MyDrive/LexAgent/chroma_memory



✅ Drive mounted. Base path: /content/drive/MyDrive/LexAgent
   ChromaDB path: /content/drive/MyDrive/LexAgent/chroma_corpus
   BM25 index:    /content/drive/MyDrive/LexAgent/bm25_index.pkl


In [8]:
# ══════════════════════════════════════════════════════
# Cell 3: Load Mistral-7B (NF4 4-bit Quantization)
# ══════════════════════════════════════════════════════
import torch
from src.models.load_model import load_mistral_7b

print(f"GPU: {torch.cuda.get_device_name(0)}")
# FIX: Changed total_mem to total_memory
print(f"VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print(f"VRAM used (before model): {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print()

model, tokenizer, llm_pipeline = load_mistral_7b()

print(f"\n✅ Mistral-7B loaded in NF4 4-bit.")
print(f"   VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

GPU: Tesla T4
VRAM total: 15.64 GB
VRAM used (before model): 0.00 GB

[LexAgent] 2026-06-15 05:00:12 | src.models.load_model | INFO | CUDA available: Tesla T4


INFO:src.models.load_model:CUDA available: Tesla T4


[LexAgent] 2026-06-15 05:00:12 | src.models.load_model | INFO | Loading mistralai/Mistral-7B-Instruct-v0.3 with NF4 4-bit quantization...


INFO:src.models.load_model:Loading mistralai/Mistral-7B-Instruct-v0.3 with NF4 4-bit quantization...
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

...

[LexAgent] 2026-06-15 05:04:12 | src.models.load_model | INFO | Model loaded. VRAM used: 4.14 GB


INFO:src.models.load_model:Model loaded. VRAM used: 4.14 GB
[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Model loaded. VRAM used: 4.14 GB
[LexAgent] 2026-06-15 05:04:12 | src.models.load_model | INFO | LLM pipeline ready (max_new_tokens=512, temp=0.1).


/content/drive/MyDrive/LexAgent/lexagent/src/models/load_model.py:117: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm_pipeline = HuggingFacePipeline(pipeline=text_pipeline)
INFO:src.models.load_model:LLM pipeline ready (max_new_tokens=512, temp=0.1).



✅ Mistral-7B loaded in NF4 4-bit.
   VRAM used: 4.14 GB


In [9]:
# ══════════════════════════════════════════════════════
# Cell 4: Load Legal-BERT Embedder (CPU)
# ══════════════════════════════════════════════════════
from src.models.load_model import load_embedder

embedder = load_embedder()

# Quick sanity check
test_embed = embedder.encode(["test query"])
print(f"\n✅ Legal-BERT embedder loaded on CPU.")
print(f"   Embedding dimension: {test_embed.shape[1]}")
print(f"   VRAM (should be same as before): {torch.cuda.memory_allocated() / 1e9:.2f} GB")

[LexAgent] 2026-06-15 05:04:24 | src.models.load_model | INFO | Loading embedder: nlpaueb/legal-bert-base-uncased (on CPU)...


INFO:src.models.load_model:Loading embedder: nlpaueb/legal-bert-base-uncased (on CPU)...


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/222k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

[LexAgent] 2026-06-15 05:04:30 | src.models.load_model | INFO | Embedder loaded: nlpaueb/legal-bert-base-uncased (dim=768)


/content/drive/MyDrive/LexAgent/lexagent/src/models/load_model.py:150: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  embedder.get_sentence_embedding_dimension())
INFO:src.models.load_model:Embedder loaded: nlpaueb/legal-bert-base-uncased (dim=768)



✅ Legal-BERT embedder loaded on CPU.
   Embedding dimension: 768
   VRAM (should be same as before): 4.14 GB


In [22]:
import sys
import os

# 1. Force the exact correct path to the front of the line
base_path = '/content/drive/MyDrive/LexAgent/lexagent'
if base_path not in sys.path:
    sys.path.insert(0, base_path)

# 2. NUKE Colab's memory of anything related to 'src'
for key in list(sys.modules.keys()):
    if key.startswith('src') or key == 'config':
        del sys.modules[key]

# 3. Verify the file actually exists physically before trying to import
file_path = os.path.join(base_path, 'src', 'data', 'download_datasets.py')
if not os.path.exists(file_path):
    print(f"❌ CRITICAL ERROR: Could not find {file_path}")
    print("Please check your Google Drive to make sure the file wasn't accidentally moved.")
else:
    print("✅ File found on disk! Forcing fresh import...")

    # 4. Import and run!
    from src.data.download_datasets import download_all_datasets

    print("Starting secure download process...")
    dataset_paths = download_all_datasets(save_dir='/content/drive/MyDrive/LexAgent')

    print(f"\n🎉 ALL PHASE 1 DATASETS SECURED:")
    for name, path in dataset_paths.items():
        print(f"   {name}: {path}")

✅ File found on disk! Forcing fresh import...
Starting secure download process...
[LexAgent] 2026-06-15 05:39:44 | src.data.download_datasets | INFO | LegalBench already exists, skipping download.


INFO:src.data.download_datasets:LegalBench already exists, skipping download.


[LexAgent] 2026-06-15 05:39:44 | src.data.download_datasets | INFO | CaseHOLD already exists, skipping download.


INFO:src.data.download_datasets:CaseHOLD already exists, skipping download.


[LexAgent] 2026-06-15 05:39:44 | src.data.download_datasets | INFO | Downloading SCOTUS (coastalcph/lex_glue, first 3000 opinions)...


...

[LexAgent] 2026-06-15 05:39:53 | src.data.download_datasets | INFO | SCOTUS saved: 3000 opinions → /content/drive/MyDrive/LexAgent/scotus_dataset.json


INFO:src.data.download_datasets:SCOTUS saved: 3000 opinions → /content/drive/MyDrive/LexAgent/scotus_dataset.json


[LexAgent] 2026-06-15 05:39:53 | src.data.download_datasets | INFO | All datasets ready: ['legalbench', 'casehold', 'scotus']


INFO:src.data.download_datasets:All datasets ready: ['legalbench', 'casehold', 'scotus']



🎉 ALL PHASE 1 DATASETS SECURED:
   legalbench: /content/drive/MyDrive/LexAgent/legalbench_dataset.json
   casehold: /content/drive/MyDrive/LexAgent/casehold_dataset.json
   scotus: /content/drive/MyDrive/LexAgent/scotus_dataset.json


In [17]:
import sys
import os

print("--- DIAGNOSTIC CHECK ---")
path_1 = '/content/drive/MyDrive/LexAgent'
print(f"\nContents of {path_1}:")
try:
    print(os.listdir(path_1))
except Exception as e:
    print(f"Error: {e}")

path_2 = '/content/drive/MyDrive/LexAgent/lexagent'
print(f"\nContents of {path_2}:")
try:
    print(os.listdir(path_2))
except Exception as e:
    print(f"Error: {e}")

--- DIAGNOSTIC CHECK ---

Contents of /content/drive/MyDrive/LexAgent:
['lexagent', 'chroma_corpus', 'chroma_memory', 'legalbench_dataset.json']

Contents of /content/drive/MyDrive/LexAgent/lexagent:
['notebooks', 'requirements.txt', 'config.py', 'src', '__pycache__']


In [23]:
# ══════════════════════════════════════════════════════
# Cell 5: Download Datasets (Skip if Already on Drive)
# ══════════════════════════════════════════════════════
from src.data.download_datasets import download_all_datasets

print("Downloading datasets (will skip existing files)...\n")
dataset_paths = download_all_datasets(save_dir=DRIVE_BASE)

print(f"\n✅ All datasets ready:")
for name, path in dataset_paths.items():
    size_mb = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    print(f"   {name}: {path} ({size_mb:.1f} MB)")


[LexAgent] 2026-06-15 05:40:46 | src.data.download_datasets | INFO | LegalBench already exists, skipping download.


INFO:src.data.download_datasets:LegalBench already exists, skipping download.


[LexAgent] 2026-06-15 05:40:46 | src.data.download_datasets | INFO | CaseHOLD already exists, skipping download.


INFO:src.data.download_datasets:CaseHOLD already exists, skipping download.


[LexAgent] 2026-06-15 05:40:46 | src.data.download_datasets | INFO | SCOTUS already exists, skipping download.


INFO:src.data.download_datasets:SCOTUS already exists, skipping download.


[LexAgent] 2026-06-15 05:40:46 | src.data.download_datasets | INFO | All datasets ready: ['legalbench', 'casehold', 'scotus']


INFO:src.data.download_datasets:All datasets ready: ['legalbench', 'casehold', 'scotus']



✅ All datasets ready:
   legalbench: /content/drive/MyDrive/LexAgent/legalbench_dataset.json (0.0 MB)
   casehold: /content/drive/MyDrive/LexAgent/casehold_dataset.json (78.4 MB)
   scotus: /content/drive/MyDrive/LexAgent/scotus_dataset.json (87.1 MB)


In [24]:
# ══════════════════════════════════════════════════════
# Cell 6: Build Corpus + ChromaDB Index (Skip if Exists)
# ══════════════════════════════════════════════════════
import chromadb
from src.data.build_corpus import build_legal_corpus, build_chroma_index
from config import CORPUS_JSON_PATH, KNOWN_CASES_PATH
import json
from src.data.corpus_schema import CorpusDocument

# Check if ChromaDB index already exists
chroma_exists = os.path.exists(CHROMA_CORPUS_PATH) and os.listdir(CHROMA_CORPUS_PATH)

if chroma_exists:
    print("ChromaDB index found on Drive. Loading existing index...")
    client = chromadb.PersistentClient(path=CHROMA_CORPUS_PATH)
    collection = client.get_collection("legal_corpus")

    # Load corpus from JSON
    with open(CORPUS_JSON_PATH, 'r') as f:
        corpus_dicts = json.load(f)
    corpus = [CorpusDocument.from_dict(d) for d in corpus_dicts]

    print(f"   Loaded {collection.count()} documents from existing index.")
else:
    print("Building corpus and ChromaDB index (first run)...\n")

    # Build corpus from downloaded datasets
    corpus = build_legal_corpus(dataset_paths)

    # Build ChromaDB index with Legal-BERT embeddings
    collection = build_chroma_index(corpus, embedder)

print(f"\n✅ Corpus ready: {len(corpus)} documents")
print(f"   ChromaDB collection: {collection.count()} vectors")
print(f"   Known cases saved: {KNOWN_CASES_PATH}")

Building corpus and ChromaDB index (first run)...

[LexAgent] 2026-06-15 05:41:09 | src.data.build_corpus | INFO | Processing SCOTUS opinions...


INFO:src.data.build_corpus:Processing SCOTUS opinions...


[LexAgent] 2026-06-15 05:41:12 | src.data.build_corpus | INFO |   → SCOTUS: 3000 documents added


INFO:src.data.build_corpus:  → SCOTUS: 3000 documents added


[LexAgent] 2026-06-15 05:41:12 | src.data.build_corpus | INFO | Processing CaseHOLD holdings...


...

[LexAgent] 2026-06-15 06:39:59 | src.data.build_corpus | INFO | ChromaDB index built: 3000 documents, 2793 known case names


INFO:src.data.build_corpus:ChromaDB index built: 3000 documents, 2793 known case names


[LexAgent] 2026-06-15 06:39:59 | src.data.build_corpus | INFO | Known cases saved: 2793 entries → /content/drive/MyDrive/LexAgent/known_cases.pkl


INFO:src.data.build_corpus:Known cases saved: 2793 entries → /content/drive/MyDrive/LexAgent/known_cases.pkl



✅ Corpus ready: 3000 documents
   ChromaDB collection: 3000 vectors
   Known cases saved: /content/drive/MyDrive/LexAgent/known_cases.pkl


In [25]:
# ══════════════════════════════════════════════════════
# Cell 7: Build BM25 Index (Skip if Exists)
# ══════════════════════════════════════════════════════
from src.retrieval.bm25_retriever import BM25Retriever
from config import BM25_INDEX_PATH

if os.path.exists(BM25_INDEX_PATH):
    print("BM25 index found on Drive. Loading...")
    bm25_retriever = BM25Retriever.load_index(BM25_INDEX_PATH, corpus)
else:
    print("Building BM25 index (first run)...")
    bm25_retriever = BM25Retriever(corpus)
    bm25_retriever.save_index(BM25_INDEX_PATH)

print(f"\n✅ BM25 index ready: {len(corpus)} documents indexed")
print(f"   Saved at: {BM25_INDEX_PATH}")

Building BM25 index (first run)...
[LexAgent] 2026-06-15 06:40:01 | src.retrieval.bm25_retriever | INFO | Building BM25 index over 3000 documents...


INFO:src.retrieval.bm25_retriever:Building BM25 index over 3000 documents...


[LexAgent] 2026-06-15 06:40:01 | src.retrieval.bm25_retriever | INFO | BM25 index built successfully.


INFO:src.retrieval.bm25_retriever:BM25 index built successfully.


[LexAgent] 2026-06-15 06:40:02 | src.retrieval.bm25_retriever | INFO | BM25 index saved: /content/drive/MyDrive/LexAgent/bm25_index.pkl (7.62 MB)


INFO:src.retrieval.bm25_retriever:BM25 index saved: /content/drive/MyDrive/LexAgent/bm25_index.pkl (7.62 MB)



✅ BM25 index ready: 3000 documents indexed
   Saved at: /content/drive/MyDrive/LexAgent/bm25_index.pkl


In [26]:
# ══════════════════════════════════════════════════════
# Cell 8: Test Hybrid Retrieval with Sample Query
# ══════════════════════════════════════════════════════
from src.retrieval.dense_retriever import DenseRetriever
from src.retrieval.hybrid_retriever import HybridRetriever

# Initialize retrievers
dense_retriever = DenseRetriever(collection, embedder)
hybrid_retriever = HybridRetriever(dense_retriever, bm25_retriever)

# ── Test query ──────────────────────────────────────────
query = "Can a landlord evict a tenant without notice?"
results = hybrid_retriever.retrieve(query, jurisdiction="US_Federal")

print(f"\n{'═' * 70}")
print(f"HYBRID RETRIEVAL RESULTS")
print(f"Query: {query}")
print(f"Jurisdiction: US_Federal")
print(f"{'═' * 70}\n")

# Print top-3 results
for i, doc in enumerate(results[:3], 1):
    rrf_score = doc.metadata.get('rrf_score', 'N/A')
    print(f"── Result {i} ──")
    print(f"   Case Name:  {doc.case_name if doc.case_name else '(not extracted)'}")
    print(f"   Source:     {doc.source}")
    print(f"   Court:     {doc.court if doc.court else '(unknown)'}")
    print(f"   RRF Score: {rrf_score}")
    print(f"   Text:      {doc.text[:200]}...")
    print()

# ── Test entity map (NOVEL: feeds CCE) ───────────────────
entity_map = hybrid_retriever.get_entity_map(results)
print(f"\n── Entity Map (for CCE) ──")
print(f"   Known case names in results: {len(entity_map)}")
for case_name in list(entity_map.keys())[:5]:
    print(f"   • {case_name}")

print(f"\n✅ Hybrid retrieval working correctly!")

[LexAgent] 2026-06-15 06:40:02 | src.retrieval.dense_retriever | INFO | DenseRetriever initialized (top_k=4, collection=legal_corpus)


INFO:src.retrieval.dense_retriever:DenseRetriever initialized (top_k=4, collection=legal_corpus)


[LexAgent] 2026-06-15 06:40:02 | src.retrieval.hybrid_retriever | INFO | HybridRetriever initialized (RRF k=60)


INFO:src.retrieval.hybrid_retriever:HybridRetriever initialized (RRF k=60)


[LexAgent] 2026-06-15 06:40:02 | src.retrieval.dense_retriever | INFO | Dense retrieval: query='Can a landlord evict a tenant without notice?...' → 4 results


...

[LexAgent] 2026-06-15 06:40:02 | src.retrieval.hybrid_retriever | INFO | Hybrid retrieval: query='Can a landlord evict a tenant without notice?...' → 7 results (from 4 dense + 4 BM25)


INFO:src.retrieval.hybrid_retriever:Hybrid retrieval: query='Can a landlord evict a tenant without notice?...' → 7 results (from 4 dense + 4 BM25)



══════════════════════════════════════════════════════════════════════
HYBRID RETRIEVAL RESULTS
Query: Can a landlord evict a tenant without notice?
Jurisdiction: US_Federal
══════════════════════════════════════════════════════════════════════

── Result 1 ──
   Case Name:  As we stated in Bowles v. Seminole Rock Co
   Source:     SCOTUS
   Court:     Supreme Court of the United States
   RRF Score: 0.03225806451612903
   Text:      393 U.S. 268
89 S.Ct. 518
21 L.Ed.2d 474
Joyce C. THORPE, Petitioner,v.HOUSING AUTHORITY OF the CITY OF DURHAM.
No. 20.
Argued Oct. 23, 1968.
Decided Jan. 13, 1969.

James M. Nabrit, III, New York Cit...

── Result 2 ──
   Case Name:  Appellant in support of its contentions presses Michigan Public Utilities Commission v. Duke
   Source:     SCOTUS
   Court:     Supreme Court of the United States
   RRF Score: 0.01639344262295082
   Text:      341 U.S. 105
71 S.Ct. 601
95 L.Ed. 788
CALIFORNIA STATE AUTO. ASS'N INTER-INSURANCE BUREAUv.MALONEY, Insurance Com

INFO:src.retrieval.hybrid_retriever:Entity map built: 7 case names from 7 documents



── Entity Map (for CCE) ──
   Known case names in results: 7
   • As we stated in Bowles v. Seminole Rock Co
   • Appellant in support of its contentions presses Michigan Public Utilities Commission v. Duke
   • Cf. United States v. Gilbert Associates
   • First Amendment, applicable to the States by reason of the Fourteenth Amendment. The case differs from Braunfeld v. Brown
   • DeBacker v. Brainard

✅ Hybrid retrieval working correctly!


In [29]:
# ══════════════════════════════════════════════════════
# Cell 9: System Summary
# ══════════════════════════════════════════════════════
import pickle
import os
import torch
from config import BM25_INDEX_PATH, KNOWN_CASES_PATH, MODEL_NAME, EMBEDDER_NAME

print("\n" + "═" * 70)
print(" LexAgent v2.0 — Phase 1 Summary")
print("═" * 70)

# Corpus stats
scotus_count = sum(1 for d in corpus if d.source == 'SCOTUS')
casehold_count = sum(1 for d in corpus if d.source == 'CaseHOLD')
print(f"\n📚 Corpus:")
print(f"   Total documents:    {len(corpus)}")
print(f"   SCOTUS opinions:    {scotus_count}")
print(f"   CaseHOLD holdings:  {casehold_count}")

# Index stats
print(f"\n🔍 Indices:")
print(f"   ChromaDB vectors:   {collection.count()}")
bm25_size = os.path.getsize(BM25_INDEX_PATH) / 1e6 if os.path.exists(BM25_INDEX_PATH) else 0
print(f"   BM25 index size:    {bm25_size:.1f} MB")

# Known cases
if os.path.exists(KNOWN_CASES_PATH):
    with open(KNOWN_CASES_PATH, 'rb') as f:
        known_cases = pickle.load(f)
    print(f"   Known case names:   {len(known_cases)}")

# VRAM stats
print(f"\n💾 VRAM:")
print(f"   Model VRAM:         {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   Peak VRAM:          {torch.cuda.max_memory_allocated() / 1e9:.2f} GB")
# FIX: Changed total_mem to total_memory
print(f"   Total GPU memory:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Model info
print(f"\n🤖 Models:")
print(f"   LLM:       {MODEL_NAME} (NF4 4-bit)")
print(f"   Embedder:  {EMBEDDER_NAME} (CPU)")

print(f"\n{'═' * 70}")
print(f"✅ Phase 1 complete! Ready for Phase 2 (DVA+ Debate Pipeline).")
print(f"{'═' * 70}")


══════════════════════════════════════════════════════════════════════
 LexAgent v2.0 — Phase 1 Summary
══════════════════════════════════════════════════════════════════════

📚 Corpus:
   Total documents:    3000
   SCOTUS opinions:    3000
   CaseHOLD holdings:  0

🔍 Indices:
   ChromaDB vectors:   3000
   BM25 index size:    7.6 MB
   Known case names:   2793

💾 VRAM:
   Model VRAM:         4.14 GB
   Peak VRAM:          4.25 GB
   Total GPU memory:   15.64 GB

🤖 Models:
   LLM:       mistralai/Mistral-7B-Instruct-v0.3 (NF4 4-bit)
   Embedder:  nlpaueb/legal-bert-base-uncased (CPU)

══════════════════════════════════════════════════════════════════════
✅ Phase 1 complete! Ready for Phase 2 (DVA+ Debate Pipeline).
══════════════════════════════════════════════════════════════════════
